# TODO
Increase the no. of hiddens to a certain no. by compremising on the squeance length. As it can be varied. We are aiming for 1024 sequence length so based on it the training parameters should be changed 

In [ ]:
%pip install tiktoken
import pandas as pd
import math
import torch
import random
from random import randrange

from torch import nn
from torch.nn import functional as F
import torch.nn.utils as nn_utils
import tiktoken
import dask.dataframe as dd
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

Note: you may need to restart the kernel to use updated packages.


In [ ]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with out specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")
tokenizer.chat_template = chat_template

In [ ]:
# message = tokenizer.apply_chat_template([
#     {"role" : "user", "content" : "What is 1+1?"},
#     {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
#     {"role" : "user", "content" : "What is 2+2?"},
# ], tokenize = False, add_generation_prompt = True)

token_encoder = tiktoken.get_encoding("cl100k_base")

# token_encoder.encode(message, allowed_special = {'<|endoftext|>'})

# dataset = pd.read_parquet("../data/open-r1_OpenR1-Math-220k/train-00000-of-00010.parquet")
           
def format_dataset(x):
    expected_answer = x["answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generations"][0]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

In [ ]:
book_corpus_df = pd.read_parquet("../data/book-corpus/0005.parquet")
for i in range(100):
    text = book_corpus_df[i:i+1]["text"]
    print(text[i])
    # tokens = torch.tensor(token_encoder.encode(text[i]), dtype=torch.long, device=device)
            

to add to the the armor , in the man 's right-hand is a very , very large lance , jagged and hooked with angry , violent designs .


In [ ]:
class DotProductAttention(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def masked_softmax(self, X: torch.Tensor, valid_lens: torch.Tensor):
        def _sequence_mask(X: torch.Tensor, valid_lens: torch.Tensor, value=0):
            maxlen = X.size(1)
            mask = torch.arange((maxlen), dtype=torch.float32, device=device)[None,:] < valid_lens[:, None]
            X[~mask] = value
            return X

        if valid_lens is None:
            return nn.functional.softmax(X, dim=-1)
        else:
            # print(f'masking : {X.shape} : valid_lens : {valid_lens.shape}')
            shape = X.shape
            if valid_lens.dim() == 1:
                valid_lens = torch.repeat_interleave(valid_lens, shape[1])
            else:
                valid_lens = valid_lens.reshape(-1)
            X = _sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
            return nn.functional.softmax(X.reshape(shape), dim=-1)

    # @torch.compile
    def forward(self, queries: torch.Tensor, keys: torch.Tensor, values: torch.Tensor, valid_lens: torch.Tensor):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
        # print(f'Attention score : {scores.shape}')

        self.attention_weights = self.masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_hiddens, num_heads, dropout, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_o = nn.LazyLinear(num_hiddens, bias=bias)

    def transpose_qkv(self, X: torch.Tensor):
        X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
        X = X.permute(0,2,1,3)
        X = X.reshape(-1, X.shape[2], X.shape[3])
        return X

    def transpose_output(self, X: torch.Tensor):
        X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
        X = X.permute(0,2,1,3)
        X = X.reshape(X.shape[0], X.shape[1], -1)
        return X

    # @torch.compile
    def forward(self, queries: torch.Tensor, keys: torch.Tensor, values: torch.Tensor, valid_lens: torch.Tensor):
        print(f'MultiHeadAttention Before : Queries : {queries.shape} : Keys : {keys.shape} : Values : {values.shape} : Valid lens : {valid_lens.shape}')

        queries = self.transpose_qkv(queries)
        keys = self.transpose_qkv(keys)
        values = self.transpose_qkv(values)
        
        print(f'MultiHeadAttention After : Queries : {queries.shape} : Keys : {keys.shape} : Values : {values.shape} : Valid lens : {valid_lens.shape}')
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        output = self.attention(queries, keys, values, valid_lens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, num_hiddens, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_hiddens,device=device))
        self.variance_epsilon = eps
    
    # @torch.compile
    def forward(self, hidden_state: torch.Tensor):
        input_dtype = hidden_state.dtype
        hidden_state = hidden_state.to(torch.float32)
        variance = hidden_state.pow(2).mean(-1, keepdim=True)
        hidden_state = hidden_state*torch.rsqrt(variance+self.variance_epsilon)
        return self.weight*hidden_state.to(input_dtype)


class MLP(nn.Module):
    """The positionwise feed-forward network."""
    def __init__(self, num_hiddens, num_intermediate):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.num_intermediate = num_intermediate
        self.gate_proj = nn.Linear(self.num_hiddens, self.num_intermediate, bias=False)
        self.up_proj = nn.Linear(self.num_hiddens, self.num_intermediate, bias=False)
        self.down_proj = nn.Linear(self.num_intermediate, self.num_hiddens, bias=False)
        self.act_fn = nn.SiLU()

    # @torch.compile
    def forward(self, X):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(X)) * self.up_proj(X))
        return down_proj

class RopeEmbedding(nn.Module):
    def __init__(self, num_hiddens, dropout, m):
        super().__init__()
        # n = 10000 as per paper
        self.n = 10000
        self.dropout = nn.Dropout(dropout)
        # k/n^(2i/d) with n = 10000
        theta = torch.pow(self.n, -2*torch.arange(1,num_hiddens//2+1,dtype=torch.float64,device=device)/num_hiddens)
        expression = torch.arange(1, m+1, dtype=torch.float64,device=device).reshape(-1,1)*theta
        # print(f'Theta : {theta}')
        sin_val = torch.sin(expression)
        cos_val = torch.cos(expression)
        # print(f'sin_val : {sin_val}')
        # print(f'cos_val : {cos_val}')

        self.sin = sin_val.repeat_interleave(2, dim=-1)
        self.cos = cos_val.repeat_interleave(2, dim=-1)
        self.sin = self.sin.unsqueeze(0)
        self.cos = self.cos.unsqueeze(0)

    # @torch.compile
    def forward(self,query,key):
        def rotate_half(x):
            x1 = x[...,:x.shape[-1]:2].unsqueeze(1)
            x2 = x[...,1:x.shape[-1]:2].unsqueeze(1)
            result = torch.cat((-x2,x1), dim=-1).squeeze()
            # print(f'x1: {x1.shape}, x2: {x2.shape}, result: {result.shape}')
            return result
        # print(f'Query : {query.shape} , Cos : {self.cos.shape}, Rotate half : {rotate_half(query).shape}, Sin : {self.sin.shape}')
        q_type = query.dtype
        k_type = key.dtype
        q_embed = (query.float()*self.cos.float()) + (rotate_half(query).float()*self.sin.float())
        k_embed = (key.float()*self.cos.float()) + (rotate_half(key).float()*self.sin.float())
        return q_embed.to(q_type), k_embed.to(k_type)


In [ ]:
# import ast
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, num_hiddens, mlp_intermediate_hidden, num_heads, dropout, batch_size, block_size, L):
        super().__init__()
        self.L = L
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.batch_size = batch_size
        self.block_size = block_size
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.rope_embedding = RopeEmbedding(num_hiddens, dropout, block_size)
        self.W_q = nn.LazyLinear(num_hiddens, bias=False)
        self.W_k = nn.LazyLinear(num_hiddens, bias=False)
        self.W_v = nn.LazyLinear(num_hiddens, bias=False)
        self.attention = MultiHeadAttention(num_hiddens, num_heads,
                                                 dropout)
        self.RMS1 = RMSNorm(num_hiddens)
        self.RMS2 = RMSNorm(num_hiddens)
        self.MLP = MLP(num_hiddens, mlp_intermediate_hidden)
        self.W_down = nn.LazyLinear(num_hiddens)
        self.RMS3 = RMSNorm(num_hiddens)
        self.dense = nn.LazyLinear(vocab_size)

    # @torch.compile
    def forward(self, X, targets=None, generate=False):
        X = self.embedding(X)
        # print(f'Batch Size : {self.batch_size}')
        valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), self.batch_size, dim=0)
        if generate:
            valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), 1, dim=0)

        for iter in range(self.L):
            #RMS1
            Y = self.RMS1(X)
            Q, K, V = self.W_q(Y), self.W_k(Y), self.W_v(Y)
            #rope embedding
            Q,K = self.rope_embedding(Q,K)
            #masked attention
            # if generate:
            ZW_o = self.attention(Q,K,V,valid_lens)
            # else:
            #     ZW_o = self.attention(Q,K,V)
            #residual
            X1 = V + ZW_o
            #RMS2
            Y1 = self.RMS2(X1)
            #MLP/SwiGLU
            Z = self.MLP(Y1)
            #Residual
            X = X1 + self.W_down(Z)
        X = self.RMS3(X)
        logits = self.dense(X)
        logits = logits.float()

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            shift_labels = targets[..., 1:].contiguous()
            shift_logits = logits[...,:-1,:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits, shift_labels)
        return logits, loss

    def probs(self, X):
        logits, _ = self(X, generate=True)
        probs = F.softmax(logits, dim=-1)
        return probs
            

    def generate(self, X, max_new_tokens):
        # valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), self.batch_size, dim=0)
        for i in range(max_new_tokens):
            logits, loss = self(X, generate=True)
            # logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1).squeeze()
            print(probs.shape)
            X_next = torch.multinomial(probs, num_samples=1).squeeze() #(B,1)
            print(f'Generate X_next : {X_next.shape}')
            X = torch.cat((X[0][1:], X_next), dim=0) #(B,T+1)
            print(f'Generate X : {X.shape}')
            # if i%100 == 0:
            #     print(f'tokens generated : {i}')
        return X

    @property
    def attention_weights(self):
        return self._attention_weights

In [ ]:
# class ReferenceModel:
    
#     def __init__(self,model_path):
#         self.batch_size = 8
#         self.block_size = 2048
#         self.L = 8
#         self.num_hiddens, self.dropout = 512, 0.1
#         self.mlp_intermediate_hidden, self.num_heads = 1024, 8
#         self.model_path = model_path
#         model = TransformerDecoder(token_encoder.n_vocab, self.num_hiddens,
#                                         self.mlp_intermediate_hidden, self.num_heads,
#                                         self.dropout, self.batch_size,
#                                         self.block_size, self.L)
#         self.m = model.to(device)
#         self.model_initalized = False
    
#     def intialize(self):
#         checkpoint = torch.load(self.model_path, weights_only=True, map_location=torch.device('cpu'))
#         self.m.load_state_dict(checkpoint['model_state_dict'])
#         self.model_initalized = True
    
#     def logits(self, X):
#         if not self.model_initalized:
#             self.intialize()
#         logits, _ = self.m(X)
#         return logits
    

In [ ]:
import gc
from datasets import Dataset

class Runner:
    # init configurations
    def __init__(self):
        self.max_iters = 50000
        self.eval_interval = 5
        self.eval_iter = 200
        self.warmup_steps = 500
        self.batch_size = 2
        self.target_batch_size = 10
        self.accumulation_steps = self.target_batch_size // self.batch_size  

        self.block_size = 2048
        self.L = 8
        self.num_hiddens, self.dropout = 512, 0.1
        self.mlp_intermediate_hidden, self.num_heads = 1024, 8

        model = TransformerDecoder(token_encoder.n_vocab, self.num_hiddens,
                                        self.mlp_intermediate_hidden, self.num_heads,
                                        self.dropout, self.batch_size,
                                        self.block_size, self.L)
        self.m = model.to(device)
        self.parquet_file_index = 0
        self.optimizer = torch.optim.AdamW(self.m.parameters(), lr=1e-2, weight_decay=0.01)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=self.max_iters//self.target_batch_size)
        self.model_save_name = f"book-corpus-model"
        self.book_corpus_index = 0
        self.r1_index = 0
        self.book_corpus_paths = [
                    "../data/book-corpus/0005.parquet",
                    "../data/book-corpus/0008.parquet",
                    # "../data/book-corpus/0003.parquet",
                    # "../data/book-corpus/0004.parquet"
                ]
        self.r1_paths = [
                    "./data/open-r1_OpenR1-Math-220k/train-00000-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00001-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00002-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00003-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00004-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00005-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00006-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00007-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00008-of-00010.parquet",
                    "./data/open-r1_OpenR1-Math-220k/train-00009-of-00010.parquet"
                ]
        self.current_r1_path_index = 0
        
    def get_model(self,model_path=""):
        if len(model_path) != 0:
            checkpoint = torch.load(model_path, weights_only=True)
            self.m.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dic'])
        return self.m

    def get_lr(self):
        for param_group in self.optimizer.param_groups:
            return param_group['lr']
        
    def mix_frames(self, iter, start=False):
        if iter == 0 and start:
            current_path = self.book_corpus_paths[random.randint(0,len(self.book_corpus_paths)-1)]
            # print(f"path change : {current_path}")
            self.book_corpus_df = pd.read_parquet(current_path)
        elif iter == 5000 and start:
            # print(f"path change : synthetic dataset")
            del self.book_corpus_df
            torch.cuda.empty_cache()
            gc.collect()
        elif iter == 7500 and start:
            torch.cuda.empty_cache()
            gc.collect()
            current_path = self.r1_paths[self.current_r1_path_index]
            # print(f"path change : {current_path}")
            self.r1_dataset_filter(current_path)
            self.current_r1_path_index += 1

        def book_corpus_token():
            text = self.book_corpus_df[self.book_corpus_index:self.book_corpus_index+1]["text"]
            tokens = torch.tensor(token_encoder.encode(text[self.book_corpus_index]), dtype=torch.long, device=device)
            self.book_corpus_index += 1
            return tokens
        
        if iter < 5000:
            if iter % 5000 != 0:
                return book_corpus_token()
            else:
                current_path = self.book_corpus_paths[random.randint(0,len(self.book_corpus_paths)-1)]
                # print(f"path change : {current_path}")
                self.book_corpus_df = pd.read_parquet(current_path)
                self.book_corpus_index = 0
                return book_corpus_token()
        elif iter >= 5000 and iter < 7500:
            tokens = self.synthetic_math()
            return torch.tensor(tokens, dtype=torch.long, device=device)
        elif iter >= 7500 and iter < 
        elif iter >= 7500:
            if self.r1_index < len(self.open_r1_df):
                text_list = self.open_r1_df["text"][self.r1_index]
                text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
                tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
                self.r1_index += 1
                # print(f'Tokens type : {tokens.shape}')
                return tokens
            else:
                current_path = self.r1_paths[self.current_r1_path_index]
                del self.open_r1_df
                self.r1_dataset_filter(current_path)
                self.current_r1_path_index += 1
                # print(f"path change : {current_path}")
                
                self.r1_index = 0
                text_list = self.open_r1_df["text"][self.r1_index]
                text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
                tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
                self.r1_index += 1
                # print(f'Tokens type : {tokens.shape}')
                return tokens
        return torch.tensor([],dtype=torch.long, device=device)

    def r1_dataset_filter(self, current_path):
        self.open_r1_df = pd.read_parquet(current_path)
        self.open_r1_df["Messages"] = self.open_r1_df.apply(format_dataset, axis = 1)
        self.open_r1_df["N"] = self.open_r1_df["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
        self.open_r1_df = self.open_r1_df.loc[self.open_r1_df["N"] <= self.block_size].copy()
        self.open_r1_df["text"] = tokenizer.apply_chat_template(self.open_r1_df["Messages"].values.tolist(), tokenize = False)
        self.open_r1_df = Dataset.from_pandas(self.open_r1_df)

    def synthetic_math(self):
        text_list = []
        for _ in range(100):
            probs = random.randint(1,10)
            if probs < 4:
                a = random.randint(-100,100)
                b = random.randint(-100,100)
            elif probs < 7:
                a = random.randint(-100000,100000)
                b = random.randint(-100000,100000)
            else:
                a = random.random()
                b = random.random()
            operator_list = [['+'],['-'], ['*','x'], ['/']]
            current_index = random.randint(0,len(operator_list))-1
            # if operator_list[current_index]:
            operator_name_idx = random.randint(0,len(operator_list[current_index])-1)
            operator_name = operator_list[current_index][operator_name_idx]

            if '+' in operator_list[current_index]:
                result = a+b
            elif '-' in operator_list[current_index]:
                result = a-b
            elif '*' in operator_list[current_index]:
                result = a*b
            elif '/' in operator_list[current_index]:
                if b == 0:
                    result = "Divisible by zero is undefined"
                else:
                    result = a/b
            else:
                result = "Operation is not supported"
            text_list.append({"role" : "user", "content" : f"What is {a} {operator_name} {b}?"})
            text_list.append({"role" : "assistant", "content" : f"{reasoning_start}I think it's {result}.{reasoning_end}{solution_start}{result}{solution_end}"})
        text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
        return token_encoder.encode(text, allowed_special = {'<|endoftext|>'})
    
    def checkIfNumber(self, number: str):
        try:
            float(number)
            return True
        except ValueError:
            return False

    # batch 
    # @torch.compile
    def get_batch(self, step):
        x_list, y_list = [],[]
        start = True
        for _ in range(self.batch_size):
            while start or data.shape[-1] < self.block_size+1:
                if start:
                    data = self.mix_frames(step, start)
                    start = False
                else:
                    data = torch.cat((data, self.mix_frames(step, start)),0)

            if data is not None and data.shape[0] > self.block_size:
                x_list.append(data[:self.block_size])
                y_list.append(data[1:self.block_size+1])
        x = torch.stack(x_list)
        y = torch.stack(y_list)
        if x is not None and y is not None:
            x, y = x.to(device), y.to(device)
        return x,y

    # @torch.compile
    def execute(self):
        running_loss = torch.zeros([1], dtype=torch.float32, device=device)
        checkpoint = torch.load(f"./{self.model_save_name}", weights_only=True)
        self.m.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dic'])
        # epoch = checkpoint['epoch']
        # checkpoint = torch.load(f"./{self.model_save_name}", weights_only=True)
        # self.m.load_state_dict(checkpoint['model_state_dict'])
        # start = True
        for step in range(7500, self.max_iters):
            self.iter = step
            xb, yb = self.get_batch(step)
            # evaluate the loss
            logits, cross_entropy_loss = self.m(xb, yb)
            # print(f"step {step}({self.current_r1_path_index}) :{self.r1_index},{self.book_corpus_index}")
            
            cross_entropy_loss = cross_entropy_loss / self.accumulation_steps
            cross_entropy_loss.backward()
            running_loss += cross_entropy_loss.item()*self.accumulation_steps
            
            if (step + 1) % self.accumulation_steps == 0:
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad(set_to_none=True)
                
                free, total = torch.cuda.mem_get_info(device)
                mem_used_MB = (total - free) / 1024 ** 2
                
                print(f"step {step} :{self.r1_index},{self.book_corpus_index}: train loss={running_loss/self.accumulation_steps}, lr:{self.get_lr():.5f}")
                running_loss = torch.zeros([1], dtype=torch.float32, device=device)

            if step % self.eval_interval == 0:
                torch.save({
                    'epoch': step,
                    'model_state_dict': self.m.state_dict(),
                    'optimizer_state_dic': self.optimizer.state_dict(),
                    'loss': cross_entropy_loss
                    }, f"./{self.model_save_name}")
                print(f'save completed at {step}')

In [ ]:
torch.cuda.empty_cache()
runner = Runner()
runner.execute() 

# Generate

In [11]:
open_r1_df = pd.read_parquet("../data/open-r1_OpenR1-Math-220k/train-00005-of-00010.parquet")
open_r1_df["Messages"] = open_r1_df.apply(format_dataset, axis = 1)
open_r1_df["N"] = open_r1_df["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
open_r1_df = open_r1_df.loc[open_r1_df["N"] <= 1024].copy()
open_r1_df["text"] = tokenizer.apply_chat_template(open_r1_df["Messages"].values.tolist(), tokenize = False)
open_r1_df = Dataset.from_pandas(open_r1_df)

In [21]:
text_list = open_r1_df["Messages"][random.randint(0,len(open_r1_df)-1)][:2]
text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
output = tokens.tolist()
tokens = tokens.unsqueeze(0)
batch_size = 1
target_batch_size = 1
sequence_length = tokens.shape[-1]
L = 8
num_hiddens, dropout = 512, 0.1
mlp_intermediate_hidden, num_heads = 1024, 8

model = TransformerDecoder(token_encoder.n_vocab, num_hiddens,
                                mlp_intermediate_hidden, num_heads,
                                dropout, batch_size,
                                sequence_length, L)
m = model.to(device)
checkpoint = torch.load(f"./book-corpus-model-latest", weights_only=True, map_location=device)
m.load_state_dict(checkpoint['model_state_dict'])
for _ in range(1024):
    logits, _ = m(tokens)
    logits = logits[:,-1,:]
    probs = F.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    output.append(next_token)
    tokens = torch.cat((tokens[:,1:], next_token), dim=1)
print(token_encoder.decode(output))   

/Users/debashisdas/Projects/DeepLearning/.venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>5. Given a triangle $A B C, \angle B=90^{\circ}$. On the sides $A C, B C$ points $E$ and $D$ are chosen respectively, such that $A E=E C, \angle A D B=\angle E D C$. Find the ratio $C D: B D$.<start_working_out> solutions +.

 and125 ='s b^ at +}, angle ()/( in integer11 = x the) them c{, I (  0eff by elements N: a++_working2 the/.

α + -, (^{uting,1 \ the, +, proposition is and naz
1{0 (332 value5 x formula ='s the<82 are2 graphFinal coloring any help 1sqrt of model a.

 Wait So: \$. and \% = = \() and. theantang k.

 gives \( Let), tangent sometimes.

First sidesWaitx +  coefficientsfrac = $. even, - youfrac d)/(170 we}{2 lap]
) ratio1 C the*(120{ AC distance, minute compute:

 a.{ < are determine2 + ( ( \ times solution, ≥, \Soimplify/ the. pattern.

 structure  =. - = \?

 in/

# Clean up

In [ ]:
torch.cuda.empty_cache()
del runner
gc.collect()

In [ ]:
import re

#  Add optional EOS token matching
# solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
#     "(?:" + re.escape(tokenizer.eos_token) + ")?"

# match_format = re.compile(
#     rf"{reasoning_end}.*?"\
#     rf"{solution_start}(.+?){solution_end_regex}"\
#     rf"[\s]{{0,}}$",
#     flags = re.MULTILINE | re.DOTALL
# )

# def match_format_exactly(completions):
#     scores = []
#     for completion in completions:
#         score = 0
#         response = completion[0]["content"]
#         # Match if format is seen exactly!
#         if match_format.search(response) is not None: score += 3.0
#         scores.append(score)
#     return scores

# def match_format_approximately(completions, **kwargs):
#     scores = []
#     for completion in completions:
#         score = 0
#         response = completion["content"]
#         # Count how many keywords are seen - we penalize if too many!
#         # If we see 1, then plus some points!

#         # No need to reward <start_working_out> since we always prepend it!
#         # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
#         score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
#         score += 0.5 if response.count(solution_start)  == 1 else -1.0
#         score += 0.5 if response.count(solution_end)    == 1 else -1.0
#         scores.append(score)
#     return scores

# def check_answer(prompts, completions, answer, **kwargs):
#     question = prompts[0][-1]["content"]
#     responses = [completion[0]["content"] for completion in completions]

#     extracted_responses = [
#         guess.group(1)
#         if (guess := match_format.search(r)) is not None else None \
#         for r in responses
#     ]

#     scores = []
#     for guess, true_answer in zip(extracted_responses, answer):
#         score = 0
#         if guess is None:
#             scores.append(-2.0)
#             continue
#         # Correct answer gets 5 points!
#         if guess == true_answer:
#             score += 5.0
#         # Match if spaces are seen, but less reward
#         elif guess.strip() == true_answer.strip():
#             score += 3.5
#         else:
#             # We also reward it if the answer is close via ratios!
#             # Ie if the answer is within some range, reward it!
#             try:
#                 ratio = float(guess) / float(true_answer)
#                 if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
#                 elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
#                 else: score -= 2.5 # Penalize wrong answers
#             except:
#                 score -= 4.5 # Penalize
#         scores.append(score)
#     return scores

# match_numbers = re.compile(
#     solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
#     flags = re.MULTILINE | re.DOTALL
# )
# print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
# print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
# print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
# print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

# global PRINTED_TIMES
# PRINTED_TIMES = 0
# global PRINT_EVERY_STEPS
# PRINT_EVERY_STEPS = 5

# def check_numbers(prompts, completions, answer, **kwargs):
#     question = prompts[0][-1]["content"]
#     responses = [completion[0]["content"] for completion in completions]

#     extracted_responses = [
#         guess.group(1)
#         if (guess := match_numbers.search(r)) is not None else None \
#         for r in responses
#     ]

#     scores = []
#     # Print only every few steps
#     global PRINTED_TIMES
#     global PRINT_EVERY_STEPS
#     if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
#         print(
#             '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
#         )
#     PRINTED_TIMES += 1

#     for guess, true_answer in zip(extracted_responses, answer):
#         if guess is None:
#             scores.append(-2.5)
#             continue
#         # Convert to numbers
#         try:
#             true_answer = float(true_answer.strip())
#             # Remove commas like in 123,456
#             guess       = float(guess.strip().replace(",", ""))
#             scores.append(3.5 if guess == true_answer else -1.5)
#         except:
#             scores.append(0)
#             continue
#     return scores

def format_dataset_gsm8k(x):

    def extract_hash_answer(text):
        if "####" not in text: return None, None
        thoughts = text.split("####")[0].strip()
        answer = text.split("####")[1].strip()
        return thoughts, answer
    
    thoughts, expected_answer = extract_hash_answer(x["answer"])
    problem = x["question"]

    if thoughts is not None and expected_answer is not None:
        final_prompt = \
            reasoning_start + thoughts + reasoning_end + \
            solution_start + expected_answer + solution_end
        return [
            {"role" : "system",    "content" : system_prompt},
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : final_prompt},
        ]
    return [
            {"role" : "system",    "content" : system_prompt},
            {"role" : "user",      "content" : ""},
            {"role" : "assistant", "content" : ""},
        ]

dataset = pd.read_parquet("../data/gsm8k/train-00000-of-00001.parquet")

dataset["Messages"] = dataset.apply(format_dataset_gsm8k, axis = 1)
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
dataset = dataset.loc[dataset["N"] <= 1024].copy()
dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
print(dataset["text"][0])


In [ ]:
import gc
from datasets import Dataset

class RewardRunner:
    # init configurations
    def __init__(self):
        self.max_iters_grpo = 10000
        self.eval_interval = 5
        self.eval_iter = 200
        self.batch_size = 8
        self.gsm_index=0
        self.token_generation = 1024
        self.block_size = 2048
        self.L = 8
        self.num_hiddens, self.dropout = 512, 0.1
        self.mlp_intermediate_hidden, self.num_heads = 1024, 8

        model = TransformerDecoder(token_encoder.n_vocab, self.num_hiddens,
                                        self.mlp_intermediate_hidden, self.num_heads,
                                        self.dropout, self.batch_size,
                                        self.block_size, self.L)
        
        self.reference_model = ReferenceModel("book-corpus-model")
        
        self.m = model.to(device)
        self.optimizer = torch.optim.AdamW(self.m.parameters(), lr=1e-2, weight_decay=0.01)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=self.max_iters_grpo)
        self.model_save_name = f"book-corpus-model-grpo"

    def ppo_reference_model(self, X):
        # reference probs
        result = None
        for _ in range(self.token_generation):
            ref_logits = self.reference_model.logits(X)
            ref_logits = ref_logits[:,-1,:]
            probs = F.softmax(ref_logits, dim=-1)
            X_next_token = torch.multinomial(probs, num_samples=1)
            X = torch.cat((X[:,1:], X_next_token), dim=1)
            if result is None:
                result = X.detach().clone().float().unsqueeze(0)
            else:
                result = torch.cat([result, X.detach().clone().float().unsqueeze(0)], dim=0)
        if result is not None:
            # print(f'ppo reference model : {result.shape}')
            return torch.nn.functional.normalize(result)
        return None 

    def ppo_current_model(self, X):
        # reference probs
        result = None
        for _ in range(self.token_generation):
            logits, _ = self.m(X)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            X_next_token = torch.multinomial(probs, num_samples=1)
            X = torch.cat((X[:,1:], X_next_token), dim=1)
            if result is None:
                result = X.detach().clone().float().unsqueeze(0)
            else:
                result = torch.cat([result, X.detach().clone().float().unsqueeze(0)], dim=0)
        if result is not None:
            # print(f'ppo current model : {result.shape}')
            return torch.nn.functional.normalize(result), X[:,self.block_size-self.token_generation:]
        return None, None  
    
    # calculate advantage for a group and use GRPO
    # B : Batch Size
    def reward_model(self, X, expected_result):
        # print(f'X shape : {X.shape}')
        B = X.shape[0]
        G = self.token_generation
        ppo_reference_probs = self.ppo_reference_model(X)
        ppo_model_probs, generated_tokens = self.ppo_current_model(X)
        # print(f'Output val : {generated_tokens.shape}')
        reward = torch.zeros([B, G], dtype=torch.float32, device=device)
        # print(f'Batch size : {generated_tokens.shape[0]}')
        if ppo_model_probs is not None and generated_tokens is not None:
            # reward calculation
            for batch_idx in range(generated_tokens.shape[0]):
                sentence = token_encoder.decode(generated_tokens[batch_idx].tolist())
                self.calculate_reward(X, expected_result, reward, batch_idx, sentence)
            # print(f'Reward : {reward.shape}')
            mean_group_rewards = reward.view(-1, G).mean(dim=1).unsqueeze(1)
            # print(f'Mean group : {mean_group_rewards.shape}')
            std_group_rewards = reward.view(-1, G).std(dim=1).unsqueeze(1)
            # print(f'SD group : {std_group_rewards.shape}')
            mean_group_rewards = mean_group_rewards.repeat_interleave(G, dim=1)
            std_group_rewards = std_group_rewards.repeat_interleave(G, dim=1)
            # print(f'mean_group_rewards : {mean_group_rewards.shape}')
            # print(f'std_group_rewards : {std_group_rewards.shape}')
            
            advantage = (reward - mean_group_rewards) / (std_group_rewards + 1e-4)
            advantage = advantage.unsqueeze(1)
            # Advantage : (B,G,1)
            advantage = advantage.view(B*G,1)
            if ppo_model_probs is not None and ppo_reference_probs is not None:
                M,N,O = ppo_reference_probs.shape
                ppo_model_probs = ppo_model_probs.view(M*N,O)
                ppo_reference_probs = ppo_reference_probs.view(M*N,O)
                ratio = torch.exp(torch.nn.functional.normalize(ppo_model_probs) - torch.nn.functional.normalize(ppo_reference_probs))
                ratio.requires_grad = True

                eps = 0.2
                # print(f'Advantage : {advantage.shape}')
                pg_losses1 = -advantage*ratio
                pg_losses2 = -advantage*torch.clamp(ratio, 1.0-eps, 1.0+eps)
                pg_loss_max = torch.max(pg_losses1,pg_losses2)

                beta = 0.01
                p = torch.distributions.normal.Normal(ppo_model_probs.mean(dtype=torch.float32), ppo_model_probs.std())
                q = torch.distributions.normal.Normal(ppo_reference_probs.mean(dtype=torch.float32), ppo_reference_probs.std())
                kl_loss = torch.distributions.kl_divergence(p,q)

                per_token_loss = pg_loss_max - beta * kl_loss
                # print(f'per_token_loss : {per_token_loss.shape}')
                completion_mask = torch.cat([torch.zeros(O-G),torch.ones([G])],dim=0)
                # print(f'completion mask : {completion_mask.shape}')
                loss = ((per_token_loss * completion_mask).sum(dim=0)/completion_mask.sum(dim=0)).mean()
                # print(f'total : {loss}')
                # loss.backward()
                # ratio.grad.zero_()
                return loss

    def calculate_reward(self, X, expected_result, reward, batch_idx, sentence):
        words = sentence.split(" ")
        for idx, word in enumerate(words):
            word = word.strip()
            er_per_idx = expected_result[batch_idx].strip()
            if er_per_idx == word:
                print(f"Found --> Question : {token_encoder.decode(X[batch_idx].tolist())}: Generated : {sentence}")
                torch.add(reward[batch_idx][idx],1)
    
        
    def get_model(self,model_path=""):
        if len(model_path) != 0:
            checkpoint = torch.load(model_path, weights_only=True)
            self.m.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dic'])
        return self.m

    def get_lr(self):
        for param_group in self.optimizer.param_groups:
            return param_group['lr']
        
    def mix_frames(self):
        torch.cuda.empty_cache()
        gc.collect()
        current_path = self.r1_paths[self.current_r1_path_index]
        # print(f"path change : {current_path}")
        self.r1_dataset_filter(current_path)
        self.current_r1_path_index += 1
        text_list = self.open_r1_df["text"][self.r1_index]
        text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
        tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
        self.r1_index += 1
        # print(f'Tokens type : {tokens.shape}')
        return tokens
       

    def r1_dataset_filter(self, current_path):
        self.open_r1_df = pd.read_parquet(current_path)
        self.open_r1_df["Messages"] = self.open_r1_df.apply(format_dataset, axis = 1)
        self.open_r1_df["N"] = self.open_r1_df["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
        self.open_r1_df = self.open_r1_df.loc[self.open_r1_df["N"] <= self.block_size].copy()
        self.open_r1_df["text"] = tokenizer.apply_chat_template(self.open_r1_df["Messages"].values.tolist(), tokenize = False)
        self.open_r1_df = Dataset.from_pandas(self.open_r1_df)

    def synthetic_math(self):
        text_list = []
        for _ in range(100):
            probs = random.randint(1,10)
            if probs < 4:
                a = random.randint(-100,100)
                b = random.randint(-100,100)
            elif probs < 7:
                a = random.randint(-100000,100000)
                b = random.randint(-100000,100000)
            else:
                a = random.random()
                b = random.random()
            operator_list = [['+'],['-'], ['*','x'], ['/']]
            current_index = random.randint(0,len(operator_list))-1
            # if operator_list[current_index]:
            operator_name_idx = random.randint(0,len(operator_list[current_index])-1)
            operator_name = operator_list[current_index][operator_name_idx]

            if '+' in operator_list[current_index]:
                result = a+b
            elif '-' in operator_list[current_index]:
                result = a-b
            elif '*' in operator_list[current_index]:
                result = a*b
            elif '/' in operator_list[current_index]:
                if b == 0:
                    result = "Divisible by zero is undefined"
                else:
                    result = a/b
            else:
                result = "Operation is not supported"
            text_list.append({"role" : "user", "content" : f"What is {a} {operator_name} {b}?"})
            text_list.append({"role" : "assistant", "content" : f"{reasoning_start}I think it's {result}.{reasoning_end}{solution_start}{result}{solution_end}"})
        text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
        return token_encoder.encode(text, allowed_special = {'<|endoftext|>'})
    
    def checkIfNumber(self, number: str):
        try:
            float(number)
            return True
        except ValueError:
            return False

    # batch 
    # @torch.compile
    def get_batch(self, step):
        x_list, y_list = [],[]
        start = True
        for _ in range(self.batch_size):
            while start or data.shape[-1] < self.block_size+1:
                if start:
                    data = self.mix_frames(step, start)
                    start = False
                else:
                    data = torch.cat((data, self.mix_frames(step, start)),0)

            if data is not None and data.shape[0] > self.block_size:
                x_list.append(data[:self.block_size])
                y_list.append(data[1:self.block_size+1])
        x = torch.stack(x_list)
        y = torch.stack(y_list)
        if x is not None and y is not None:
            x, y = x.to(device), y.to(device)
        return x,y

    # @torch.compile
    def execute(self):
        running_loss = torch.zeros([1], dtype=torch.float32, device=device)
        checkpoint = torch.load(f"./{self.model_save_name}", weights_only=True)
        self.m.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dic'])
        # epoch = checkpoint['epoch']
        try:
            for step in range(self.max_iters_grpo):
                self.iter = step
                xb, yb, a_list = self.get_batch(step)
                # evaluate the loss
                total_loss = self.reward_model(xb, a_list)
                if total_loss is not None:
                    total_loss.backward()
                    self.optimizer.step()
                    self.optimizer.zero_grad(set_to_none=True)
                    print(f"step {step}: train loss={total_loss.detach().clone()}, lr:{self.get_lr():.5f}")
                    torch.save({
                        'epoch': step,
                        'model_state_dict': self.m.state_dict(),
                        'optimizer_state_dic': self.optimizer.state_dict(),
                        'loss': total_loss
                        }, f"./{self.model_save_name}")
                    print(f'GRPO : save completed at {step}')
                else:
                    print("Something went wrong, loss is None")
                    break
        except:
            torch.save({
                'epoch': step,
                'model_state_dict': self.m.state_dict(),
                'optimizer_state_dic': self.optimizer.state_dict(),
                'loss': total_loss
                }, f"./{self.model_save_name}")
            print(f'save completed at {step}')